<a href="https://colab.research.google.com/github/xiaofanpaiooo-code/whisper-lora/blob/main/%E5%9F%BA%E4%BA%8E%E6%B7%B1%E5%BA%A6%E5%AD%A6%E4%B9%A0%E7%9A%84%E6%99%BA%E8%83%BD%E8%AF%AD%E9%9F%B3%E5%AD%97%E5%B9%95%E7%B3%BB%E7%BB%9F1.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Cell 1] 环境准备与模型加载

In [1]:
!pip install datasets transformers librosa soundfile

In [1]:
import os
# 设置环境变量，将下载超时从默认值显著提高到 100 秒
os.environ["HTTPX_TIMEOUT"] = "100.0"
os.environ["HUGGINGFACE_HUB_READ_TIMEOUT"] = "100"

In [13]:
pip install bitsandbytes accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [2]:
# 确保已安装必要的库：!pip install datasets transformers librosa soundfile
from transformers import WhisperProcessor
from datasets import load_dataset, interleave_datasets, Audio

# 加载特征提取器与分词器 (以 whisper-small 为例)
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="English",
    task="transcribe"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[Cell 2] 数据处理

In [3]:
from datasets import load_dataset, interleave_datasets, Audio

# 1. 开启流式加载，彻底规避 OOM
ds_65h = load_dataset("gongqingyu/bishe_whisper_dataset_65h", split="train", streaming=True)
ds_35h = load_dataset("gongqingyu/bishe_whisper_dataset_35h2", split="train", streaming=True)

# ================= 核心修复：合并前强制对齐特征 Schema =================
# 步骤 A：强行将两个流的音频特征空间锚定为 16000Hz（Whisper的标准输入频率）
ds_65h = ds_65h.cast_column("audio", Audio(sampling_rate=16000))
ds_35h = ds_35h.cast_column("audio", Audio(sampling_rate=16000))

# 步骤 B：文本标签列名对齐 (⚠️ 强烈预警：Schema 必须100%一致)
# 假设你在构建数据时，ds_65h 的文本叫 "text"，而 ds_35h 叫 "transcription"
# 这里提供一个健壮的预处理，先检查列名并统一改名为 "transcription"
def align_text_column(dataset):
    column_names = list(dataset.features.keys())
    if "text" in column_names and "transcription" not in column_names:
        return dataset.rename_column("text", "transcription")
    elif "sentence" in column_names and "transcription" not in column_names:
        return dataset.rename_column("sentence", "transcription")
    return dataset

ds_65h = align_text_column(ds_65h)
ds_35h = align_text_column(ds_35h)

# ================= 移除无关的冗余列 (精简内存管线) =================
# 不同数据集可能带有特有的额外字段（如 client_id, up_votes 等），这些会导致 Schema 依然不匹配
# 我们只保留 ASR 所需的核心列：'audio' 和 'transcription'
columns_to_keep = ["audio", "transcription"]
ds_65h = ds_65h.select_columns(columns_to_keep)
ds_35h = ds_35h.select_columns(columns_to_keep)
# ====================================================================

# 2. 动态交替混合数据集 (此时 Schema 已处于绝对安全的强制一致状态)
mixed_dataset = interleave_datasets([ds_65h, ds_35h], probabilities=[0.65, 0.35], seed=42)

# 3. 局部缓冲区打乱
# 缓冲区设为1000，保障每个 mini-batch 存在领域数据与通用数据的黄金混合比
shuffled_dataset = mixed_dataset.shuffle(seed=42, buffer_size=1000)

print("✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！


[Cell 3]定义特征映射管线

In [4]:
def prepare_dataset(batch):
    # 1. 批次解包：此时 batch["audio"] 是一个 list，里面包含多个音频字典
    # 使用列表推导式提取出当前 batch (16条) 所有的音频一维矩阵 array
    audio_arrays = [audio["array"] for audio in batch["audio"]]

    # 2. 批次特征提取：WhisperProcessor 天生支持传入 List[Array] 进行并行计算
    # 因为我们在 Cell 2 已经统一对齐为 16000Hz，这里可以直接写死 16000 节省提取开销
    extracted_features = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000
    )
    # 直接赋值整个批次的 input_features
    batch["input_features"] = extracted_features.input_features

    # 3. 批次文本分词：同样提取批次文本列表，并行 Tokenize
    text_strings = batch["transcription"]
    tokenized_labels = processor.tokenizer(text_strings)
    batch["labels"] = tokenized_labels.input_ids

    return batch

# 将向量化的映射函数应用到流式数据集中
# batched=True 保障了底层的 C++ 多线程吞吐，最大化规避 OOM 并缩短预处理时间
vectorized_ds = shuffled_dataset.map(prepare_dataset, batched=True, batch_size=16)

print("✅ 批次化特征映射函数已重新挂载！")

✅ 批次化特征映射函数已重新挂载！


[Cell 4] 特征维度验证

In [5]:
# 将 IterableDataset 转为迭代器，抓取第一个样本
sample_iterator = iter(vectorized_ds)
sample = next(sample_iterator)

# 验证核心特征
input_features = sample["input_features"]
labels = sample["labels"]

print("=== 特征映射维度验证报告 ===")
# 预期输出必须严格为: 80 x 3000
print(f"[声学特征] Log-Mel Spectrogram 维度: {len(input_features)} x {len(input_features[0])}")
print(f"[文本特征] Token IDs 长度: {len(labels)}")
print(f"[文本采样] 前 5 个 Token IDs: {labels[:5]}")
print("=============================")

=== 特征映射维度验证报告 ===
[声学特征] Log-Mel Spectrogram 维度: 80 x 3000
[文本特征] Token IDs 长度: 26
[文本采样] 前 5 个 Token IDs: [50258, 50259, 50359, 50363, 3322]


[Cell 5]数据收集器构建

In [4]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. 抽离声学特征与文本特征
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # 2. 将声学特征转换为 PyTorch Tensor (此时已经是 80x3000，无需额外 pad)
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 3. 动态填充文本标签至当前 Batch 的最大长度
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 4. 将填充位的 Token ID（通常是 processor.tokenizer.pad_token_id）替换为 -100
        # 这是避免计算 Padding Loss 的核心数学操作
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 5. Whisper 模型特有机制：如果序列起始标志位是 <|startoftranscript|>，将其裁掉，因为模型在 Decoder 端会自动添加
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# 实例化数据收集器
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

[Cell 6]验证集构建

In [7]:
# 1. 计算切分数量
TOTAL_SAMPLES = 60000
VAL_PERCENT = 0.05
NUM_VAL = int(TOTAL_SAMPLES * VAL_PERCENT) # 结果为 3000

# ================= 核心流式切分逻辑 =================
# 必须在同一个 shuffled_dataset 基础上进行 take 和 skip
# 确保 shuffled_dataset 在之前定义时已经固定了 seed=42

# 抽取前 3000 条作为验证集 (考试题)
val_dataset = shuffled_dataset.take(NUM_VAL)

# 跳过前 3000 条，剩下的 57000 条作为训练集 (课本)
train_dataset = shuffled_dataset.skip(NUM_VAL)
# ===================================================

print(f"✅ 数据切分完成：")
print(f" - 验证集 (Validation): {NUM_VAL} 条 (约 5%)")
print(f" - 训练集 (Train): {TOTAL_SAMPLES - NUM_VAL} 条 (约 95%)")

✅ 数据切分完成：
 - 验证集 (Validation): 3000 条 (约 5%)
 - 训练集 (Train): 57000 条 (约 95%)


In [13]:
# ================= 确保已经执行了切分逻辑 =================
# 如果你之前运行过下面这三行，这里不需要重复运行，放在这里是为了逻辑连贯
# NUM_VAL = 3000
# val_dataset = shuffled_dataset.take(NUM_VAL)
# train_dataset = shuffled_dataset.skip(NUM_VAL)

# ================= 核心修复：分别对训练集和验证集进行特征映射 =================
print("⏳ 正在挂载训练集特征映射流...")
vectorized_train_ds = train_dataset.map(
    prepare_dataset,
    batched=True,
    batch_size=16
)

print("⏳ 正在挂载验证集特征映射流...")
vectorized_val_ds = val_dataset.map(
    prepare_dataset,
    batched=True,
    batch_size=16
)

print("✅ 训练集与验证集的特征映射流构建完毕，变量已注入环境！")

⏳ 正在挂载训练集特征映射流...
⏳ 正在挂载验证集特征映射流...
✅ 训练集与验证集的特征映射流构建完毕，变量已注入环境！


[Cell 7]评估引擎配置

In [9]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.7 MB/s eta 0:00:00


In [5]:
import evaluate
import numpy as np

# 1. 加载纯英文工业级标准评估指标 WER
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # 2. 掩码还原：将 -100 替换回 pad_token_id，避免解码器崩溃
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # 3. 批量解码 Token 为可读的英文字符串
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # 4. [统计学与工程优化点：文本归一化]
    # 在英文 ASR 中，模型可能会输出 "Python" 而标签是 "python"。
    # 为了防止大小写带来的“虚假错误”拉高 WER，我们通常在评估前强制小写化。
    pred_str = [s.lower().strip() for s in pred_str]
    label_str = [l.lower().strip() for l in label_str]

    # 5. 计算词错误率 (WER)
    wer = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("✅ 纯英文 WER 评估引擎已成功挂载！")

✅ 纯英文 WER 评估引擎已成功挂载！


[Cell 8]INT8量化加载与模型准备

In [6]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# ================= 新增/修改的部分 =================
# 1. 定义 8-bit 量化配置 (这是新版 transformers 的标准写法)
quant_config = BitsAndBytesConfig(load_in_8bit=True)

# 2. 以量化配置加载基础模型
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small",
    quantization_config=quant_config, # 替换了原来的 load_in_8bit=True
    device_map="auto"
)
# ===================================================

# 3. 准备 k-bit 训练：冻结主干权重，并处理 LayerNorm 等层的精度补偿
model = prepare_model_for_kbit_training(model)

# 4. 配置 LoRA 矩阵注入逻辑
# 针对你的消融实验，这里 r=16 是一个稳健的起点
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # 精准注入注意力机制的投影层
    lora_dropout=0.05,
    bias="none"
)

# 5. 挂载 LoRA 适配器
model = get_peft_model(model, config)

# 打印可训练参数量，验证是否仅有 < 1% 的参数在参与训练
model.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267


[Cell 9]训练参数配置


In [7]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-lora-cs-study", # 保存路径
    per_device_train_batch_size=4,              # 显存够的话可以设为 8，不够就降为 2
    gradient_accumulation_steps=4,              # 梯度累积：模拟更大 Batch
    learning_rate=1e-3,                         # LoRA 建议使用较大初始学习率
    warmup_steps=100,
    max_steps=4000,                             # 步数可根据时间调整，先跑 4000 步看趋势
    gradient_checkpointing=True,                # 极其关键：大幅度省显存
    fp16=True,                                  # T4 必开，加速且省内存

    # ================= 核心修复点 =================
    eval_strategy="steps",                      # 旧版为 evaluation_strategy
    # ==============================================
    eval_steps=500,                             # 每 500 步在验证集上考一次试
    save_strategy="steps",
    save_steps=500,
    logging_steps=25,                           # 每 25 步打印一次训练 Loss

    # 生成配置（计算 WER 必备）
    predict_with_generate=True,
    generation_max_length=225,
    per_device_eval_batch_size=1,               # 验证时 batch size 务必设为 1，防 OOM

    # 报告设置
    report_to=["tensorboard"],                  # 建议开启 Tensorboard 实时看 Loss 曲线
    remove_unused_columns=False,
    label_names=["labels"],
)

In [2]:
!rm -rf ~/.cache/huggingface/datasets/*

备用方案：本地下载


In [3]:
import os
from datasets import load_dataset, concatenate_datasets, Audio
from transformers import WhisperProcessor

# 1. 挂载 Hugging Face 镜像（加速下载）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 2. 实例化 Processor
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="English",
    task="transcribe"
)

print("⏳ 正在全速下载数据至本地硬盘 (视网速可能需要几分钟，请耐心等待)...")
# 3. 移除 streaming=True，直接下载到 Colab 本地硬盘
ds_65h = load_dataset("gongqingyu/bishe_whisper_dataset_65h", split="train")
ds_35h = load_dataset("gongqingyu/bishe_whisper_dataset_35h2", split="train")

# 4. 强制对齐特征 Schema
ds_65h = ds_65h.cast_column("audio", Audio(sampling_rate=16000))
ds_35h = ds_35h.cast_column("audio", Audio(sampling_rate=16000))

# 统一保留的核心列 (避免其他冗余列干扰拼接)
columns_to_keep = ["audio", "transcription"]
ds_65h = ds_65h.select_columns(columns_to_keep)
ds_35h = ds_35h.select_columns(columns_to_keep)

# 5. 直接在本地硬盘拼接数据集
full_dataset = concatenate_datasets([ds_65h, ds_35h])

# 6. 一键切分训练集与验证集 (自动包含全局 Shuffle)
# 5% 作为验证集，固定 seed 保证消融实验严谨性
split_dataset = full_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
print(f"✅ 数据切分完毕！训练集: {len(train_dataset)}条, 验证集: {len(val_dataset)}条")

# 7. 定义特征映射函数
# 1. 定义动态转换函数 (注意：这里的返回值必须是个新的字典)
def prepare_dataset_on_the_fly(batch):
    # 实时抽取音频
    audio_arrays = [audio["array"] for audio in batch["audio"]]

    # CPU 实时计算特征
    extracted_features = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000
    )

    # 实时 Tokenize
    text_strings = batch["transcription"]
    tokenized = processor.tokenizer(text_strings)

    # 动态组装给 GPU 的输入
    return {
        "input_features": extracted_features.input_features,
        "labels": tokenized.input_ids
    }

print("⏳ 正在挂载动态计算图 (不消耗任何磁盘空间)...")

# 2. 核心替换：使用 with_transform 替代 map
vectorized_train_ds = train_dataset.with_transform(prepare_dataset_on_the_fly)
vectorized_val_ds = val_dataset.with_transform(prepare_dataset_on_the_fly)

print("🎉 动态数据流挂载完成！现在耗时为 0 秒，磁盘增加 0 MB！")

⏳ 正在全速下载数据至本地硬盘 (视网速可能需要几分钟，请耐心等待)...


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/43262 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/16500 [00:00<?, ? examples/s]

✅ 数据切分完毕！训练集: 56773条, 验证集: 2989条
⏳ 正在挂载动态计算图 (不消耗任何磁盘空间)...
🎉 动态数据流挂载完成！现在耗时为 0 秒，磁盘增加 0 MB！


[Cell 10]启动训练

In [8]:
from transformers import Seq2SeqTrainer

# 强制关闭缓存以进行训练
model.config.use_cache = False

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=vectorized_train_ds,
    eval_dataset=vectorized_val_ds,    # 传入你构建的 5% 验证集
    data_collator=data_collator,       # 之前定义的动态填充器
    compute_metrics=compute_metrics,   # 之前定义的 WER 计算函数

    # ================= 核心修复点 =================
    processing_class=processor.feature_extractor, # 旧版本为 tokenizer
    # ==============================================
)

# 🚀 启动训练！
print("训练开始，请密切关注第一个 500 步后的 WER 变化...")
trainer.train()

训练开始，请密切关注第一个 500 步后的 WER 变化...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Wer
500,0.813649,0.207786,0.189693


流式输出内容被截断，只能显示最后 5000 行内容。
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.

KeyboardInterrupt: 

In [9]:
import os
import zipfile

# 定义需要打包的训练输出文件夹
folder_to_zip = './whisper-small-lora-cs-study'
zip_file_name = 'whisper_lora_checkpoint.zip'

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file),
                           os.path.relpath(os.path.join(root, file),
                           os.path.join(folder_path, '..')))

if os.path.exists(folder_to_zip):
    print(f"⏳ 正在打包 {folder_to_zip} ...")
    zip_folder(folder_to_zip, zip_file_name)
    print(f"✅ 打包完成！请在左侧文件栏刷新并下载 {zip_file_name}")
else:
    print("❌ 未找到训练输出文件夹，请确认是否已经开始训练并产生了 checkpoint。")

⏳ 正在打包 ./whisper-small-lora-cs-study ...
✅ 打包完成！请在左侧文件栏刷新并下载 whisper_lora_checkpoint.zip
